## Imports

In [1]:
import os
from pathlib import Path
import json
import math
from keras import Model, layers
from keras.applications import Xception, xception
from keras.optimizers import SGD
from keras.losses import CategoricalCrossentropy
from keras.metrics import CategoricalAccuracy, AUC
from keras.callbacks import ModelCheckpoint, CSVLogger, LearningRateScheduler, EarlyStopping
from keras.backend import clear_session
from keras.utils import image_dataset_from_directory

In [2]:
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import tensorflow as tf
import tensorflow_addons as tfa

# ── GPU: memory growth ────────────────────────────────────────────────────────
# Prevents TF from reserving all VRAM at startup.
# Without this, the OS and browser might not be able to get GPU memory, in which case
# you get hard crashes.
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"GPU detected: {[g.name for g in gpus]}")
else:
    print("No GPU — running on CPU.")

# ── XLA JIT compilation ───────────────────────────────────────────────────────
# Fuses TF ops into optimised GPU kernels.
# Adds a one-time ~30-60s compilation cost on the first batch, then speeds up
# all subsequent batches. Worth it for multi-epoch training.
# tf.config.optimizer.set_jit(True)
# print("XLA JIT enabled.")

GPU detected: ['/physical_device:GPU:0']


## Model definitions

In [3]:
class TransferXception(Model):
    """
    Pre-trained Xception.
    Xception does NOT include internal rescaling — inputs must be in [-1, 1].
    We use xception.preprocess_input (maps [0,255] -> [-1,1]) directly on inputs.
    Augmentation is handled externally via tf.data.Dataset (Albumentations).
    """

    def __init__(self, num_classes, dropout_rate=0.5, **kwargs):
        super().__init__(**kwargs, name="transfer_xception")
        self.num_classes = num_classes
        self.dropout_rate = dropout_rate

        self.base = Xception(
            include_top=False,
            weights="imagenet"
        )
        self.base.trainable = False

        self.gap_layer = layers.GlobalAveragePooling2D()
        self.dropout_layer = layers.Dropout(dropout_rate)
        self.dense_layer = layers.Dense(self.num_classes, activation="softmax")

    def unfreeze_base(self, n_freeze=115):
        """
        Phase 2: unfreeze the top layers of the Xception base.
        Xception has ~134 layers — freezing the first 30 preserves low-level features.
        """
        self.base.trainable = True
        for i, layer in enumerate(self.base.layers):
            # RULE A: Freeze the first N layers (low-level features)
            if i < n_freeze:
                layer.trainable = False
            
            # RULE B: Freeze ALL Batch Normalization layers (for Stability)
            elif isinstance(layer, tf.keras.layers.BatchNormalization):
                layer.trainable = False
        frozen = sum(1 for l in self.base.layers if not l.trainable)
        total  = len(self.base.layers)
        print(f"{self.name}: {frozen}/{total} base layers frozen, {total - frozen} unfrozen")

    def get_config(self):
        config = super().get_config()
        config.update({
            "num_classes": self.num_classes,
            "dropout_rate": self.dropout_rate,
        })
        return config

    def call(self, inputs, training=False):
        # Step 1: preprocess to [-1, 1] as Xception expects
        x = xception.preprocess_input(inputs)

        # Step 2: forward through base
        x = self.base(x, training=training)

        x = self.gap_layer(x)
        x = self.dropout_layer(x, training=training)
        return self.dense_layer(x)

## Config and data loading

In [4]:
# ── Hyperparameters ─────────────────────────────────────────────────────────
# 299×299: native resolution for Xception (significant accuracy gain over 224)
# Note: ~2.9× more pixels per image — reduce batch_size if you hit OOM on GPU
IMAGE_SIZE     = (299, 299)
BATCH_SIZE     = 16       # adjust based on your GPU's VRAM (e.g., 8 or 16 for 8GB, 32+ for 16GB)
PHASE1_EPOCHS  = 25       # frozen-base head training
PHASE2_EPOCHS  = 40       # fine-tuning (EarlyStopping will cut this short)
PHASE1_LR      = 1e-3     # higher LR — only head is updating
PHASE2_LR      = 1e-5     # ~100× lower LR — prevent destroying pretrained weights
N_CLASSES      = 23

data_dir_path = Path("..\wikiart_split")
root_dir_path = Path(".")
checkpoints_folder_path = root_dir_path / "Checkpoints"
if not os.path.exists(checkpoints_folder_path):
    os.makedirs(checkpoints_folder_path)
metrics_folder_path = root_dir_path / "Metrics"
if not os.path.exists(metrics_folder_path):
    os.makedirs(metrics_folder_path)

seed = 123

# ── Dataset loading ──────────────────────────────────────────────────────────
AUTOTUNE = tf.data.AUTOTUNE

# 1. Load raw images (batched) from directories
train_ds = image_dataset_from_directory(
    data_dir_path / "train",
    label_mode="categorical",
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    crop_to_aspect_ratio=True,
    shuffle=True,
    seed=seed,
)
val_ds = image_dataset_from_directory(
    data_dir_path / "val",
    label_mode="categorical",
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    crop_to_aspect_ratio=True,
    shuffle=False,
    seed=seed,
)
test_ds = image_dataset_from_directory(
    data_dir_path / "test",
    label_mode="categorical",
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    crop_to_aspect_ratio=True,
    shuffle=False,
    seed=seed,
)

# ── Mixup ────────────────────────────────────────────────────────────────────
# Blends pairs of images and their labels proportionally.
def mixup(images, labels, alpha=0.4):
    images = tf.cast(images, tf.float32)
    batch_size = tf.shape(images)[0]
    lam = tf.random.uniform([], 0.0, alpha)
    indices = tf.random.shuffle(tf.range(batch_size))
    mixed_images = lam * images + (1.0 - lam) * tf.gather(images, indices)
    mixed_labels = lam * labels + (1.0 - lam) * tf.gather(labels, indices)
    return mixed_images, mixed_labels

# Applies mixup augmentation to the training dataset.
# We use map() to apply the mixup function to each batch of images and labels.
# The num_parallel_calls=AUTOTUNE argument allows TensorFlow to determine the optimal number of parallel calls for performance.
# Finally, we call prefetch(AUTOTUNE) to allow the dataset to fetch batches in the background while the model is training, improving performance.
train_ds_mixed = train_ds.map(mixup, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

Found 9326 files belonging to 23 classes.
Found 1992 files belonging to 23 classes.
Found 2022 files belonging to 23 classes.


## Weights

In [5]:
# Load class weights
with open('..\class_weights.json', 'r') as f:
    class_weights = json.load(f)
class_weights = {int(k): v for k, v in class_weights.items()}


## Model instantiation

In [6]:
clear_session() # Clear previous models from memory before instantiating new ones.

model = TransferXception(num_classes=N_CLASSES)

## Metrics and loss

In [7]:
def make_metrics(num_classes):
    """Return a fresh set of metric instances (metrics are stateful — each model needs its own)."""
    return [
        CategoricalAccuracy(name="accuracy"),
        AUC(multi_label=True, name="auc"),
        tfa.metrics.F1Score(num_classes=num_classes, average="macro", name="f1_score")
    ]


## Learning rate schedule

In [8]:
def make_cosine_warmup_scheduler(base_lr, total_epochs, warmup_epochs=5):
    """
    Cosine annealing with linear warmup.

    Warmup: LR ramps linearly from 0 to base_lr over the first warmup_epochs.
    This prevents the randomly initialised head from producing large gradients
    that destabilise the pretrained base at the start of training.

    Cosine decay: LR then follows a cosine curve from base_lr down to ~0.
    Finds better minima than step-decay or exponential decay in practice.
    """
    def scheduler(epoch, lr):
        if epoch < warmup_epochs:
            return base_lr * (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
        return base_lr * 0.5 * (1.0 + math.cos(math.pi * progress))
    return scheduler


## Phase 1 — Train heads with frozen base

Only the GAP + Dropout + Dense head is updated.  
The pretrained base is completely frozen.


In [9]:
print(f"\n{'='*60}")
print(f"Phase 1 training: {model.name}")
print(f"{'='*60}")


model.compile(
    optimizer=tfa.optimizers.AdamW(learning_rate=PHASE1_LR, weight_decay=1e-6),
    loss=CategoricalCrossentropy(name="loss", label_smoothing=0.1),
    metrics=make_metrics(num_classes=N_CLASSES),
)

callbacks = [
    ModelCheckpoint(
        checkpoints_folder_path / f"ckpt_phase1_{model.name}.tf",
        monitor="val_loss", save_best_only=True, verbose=1,
    ),
    CSVLogger(metrics_folder_path / f"log_phase1_{model.name}.csv"),
    LearningRateScheduler(
        make_cosine_warmup_scheduler(PHASE1_LR, PHASE1_EPOCHS, warmup_epochs=3)
    ),
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
]

history = model.fit(
    train_ds_mixed,
    validation_data=val_ds,
    epochs=PHASE1_EPOCHS,
    callbacks=callbacks,
    class_weight=class_weights,
    verbose=1,
)
phase1_fit_data = history

print("\nPhase 1 complete.")



Phase 1 training: transfer_xception
Epoch 1/25
583/583 [==============================] - ETA: 0s - loss: 2.8154 - accuracy: 0.2572 - auc: 0.6601 - f1_score: 0.2051
Epoch 1: val_loss improved from inf to 2.28885, saving model to Checkpoints\ckpt_phase1_transfer_xception.tf


INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase1_transfer_xception.tf\assets


INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase1_transfer_xception.tf\assets


583/583 [==============================] - 290s 408ms/step - loss: 2.8154 - accuracy: 0.2572 - auc: 0.6601 - f1_score: 0.2051 - val_loss: 2.2888 - val_accuracy: 0.4578 - val_auc: 0.8995 - val_f1_score: 0.4181 - lr: 3.3333e-04
Epoch 2/25
583/583 [==============================] - ETA: 0s - loss: 2.4518 - accuracy: 0.4234 - auc: 0.7322 - f1_score: 0.3432
Epoch 2: val_loss improved from 2.28885 to 1.95844, saving model to Checkpoints\ckpt_phase1_transfer_xception.tf


INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase1_transfer_xception.tf\assets


...

INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase1_transfer_xception.tf\assets


583/583 [==============================] - 89s 152ms/step - loss: 2.0976 - accuracy: 0.6064 - auc: 0.7721 - f1_score: 0.5080 - val_loss: 1.6350 - val_accuracy: 0.6601 - val_auc: 0.9561 - val_f1_score: 0.6358 - lr: 2.0254e-05
Epoch 25/25
583/583 [==============================] - ETA: 0s - loss: 2.0700 - accuracy: 0.6071 - auc: 0.7711 - f1_score: 0.5149
Epoch 25: val_loss improved from 1.63495 to 1.63440, saving model to Checkpoints\ckpt_phase1_transfer_xception.tf


INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase1_transfer_xception.tf\assets


INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase1_transfer_xception.tf\assets


583/583 [==============================] - 92s 158ms/step - loss: 2.0700 - accuracy: 0.6071 - auc: 0.7711 - f1_score: 0.5149 - val_loss: 1.6344 - val_accuracy: 0.6606 - val_auc: 0.9560 - val_f1_score: 0.6367 - lr: 5.0893e-06

Phase 1 complete.


In [10]:
phase1_eval_data = model.evaluate(
    test_ds,
    batch_size=BATCH_SIZE,
    return_dict=True,
    verbose=0
)
phase1_eval_data

{'loss': 1.634899377822876,
 'accuracy': 0.6627101898193359,
 'auc': 0.95622718334198,
 'f1_score': 0.6492078304290771}

## Phase 2 — Fine-tune unfrozen base layers

Unfreeze the top portion of the pretrained base and retrain at a much lower LR.  
Early layers learn generic features (edges, textures) that transfer well — keep them frozen.  
Later layers learn task-specific patterns — retrain these on art data.


In [11]:
print(f"\n{'='*60}")
print(f"Phase 2 fine-tuning: {model.name}")
print(f"{'='*60}")

# Unfreeze top layers — defaults are set inside each model class
model.unfreeze_base()

# Recompile at ~100× lower LR to avoid overwriting pretrained representations
model.compile(
    optimizer=tfa.optimizers.AdamW(learning_rate=PHASE2_LR, weight_decay=1e-7),
    loss=CategoricalCrossentropy(name="loss", label_smoothing=0.1),
    metrics=make_metrics(num_classes=N_CLASSES),
)

callbacks = [
    ModelCheckpoint(
        checkpoints_folder_path / f"ckpt_phase2_{model.name}.tf",
        monitor="val_loss", save_best_only=True, verbose=1,
    ),
    CSVLogger(metrics_folder_path / f"log_phase2_{model.name}.csv"),
    LearningRateScheduler(
        make_cosine_warmup_scheduler(PHASE2_LR, PHASE2_EPOCHS, warmup_epochs=2)
    ),
    # More patience in Phase 2 — improvements are smaller and slower
    EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True, verbose=1),
]

history = model.fit(
    train_ds_mixed,
    validation_data=val_ds,
    epochs=PHASE2_EPOCHS,
    callbacks=callbacks,
    class_weight=class_weights,
    verbose=1,
)
phase2_fit_data = history

print("\nPhase 2 complete.")



Phase 2 fine-tuning: transfer_xception
transfer_xception: 120/132 base layers frozen, 12 unfrozen
Epoch 1/40
583/583 [==============================] - ETA: 0s - loss: 2.0467 - accuracy: 0.6232 - auc: 0.7765 - f1_score: 0.5260
Epoch 1: val_loss improved from inf to 1.59382, saving model to Checkpoints\ckpt_phase2_transfer_xception.tf


INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase2_transfer_xception.tf\assets


INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase2_transfer_xception.tf\assets


583/583 [==============================] - 116s 193ms/step - loss: 2.0467 - accuracy: 0.6232 - auc: 0.7765 - f1_score: 0.5260 - val_loss: 1.5938 - val_accuracy: 0.6707 - val_auc: 0.9593 - val_f1_score: 0.6473 - lr: 5.0000e-06
Epoch 2/40
583/583 [==============================] - ETA: 0s - loss: 2.0179 - accuracy: 0.6286 - auc: 0.7746 - f1_score: 0.5339
Epoch 2: val_loss improved from 1.59382 to 1.56232, saving model to Checkpoints\ckpt_phase2_transfer_xception.tf


INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase2_transfer_xception.tf\assets


...

INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase2_transfer_xception.tf\assets


583/583 [==============================] - 104s 178ms/step - loss: 1.7377 - accuracy: 0.7923 - auc: 0.8005 - f1_score: 0.6701 - val_loss: 1.3247 - val_accuracy: 0.7796 - val_auc: 0.9771 - val_f1_score: 0.7575 - lr: 4.2113e-07
Epoch 37/40
583/583 [==============================] - ETA: 0s - loss: 1.7011 - accuracy: 0.8041 - auc: 0.8015 - f1_score: 0.6824
Epoch 37: val_loss improved from 1.32471 to 1.32292, saving model to Checkpoints\ckpt_phase2_transfer_xception.tf


INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase2_transfer_xception.tf\assets


INFO:tensorflow:Assets written to: Checkpoints\ckpt_phase2_transfer_xception.tf\assets


583/583 [==============================] - 103s 177ms/step - loss: 1.7011 - accuracy: 0.8041 - auc: 0.8015 - f1_score: 0.6824 - val_loss: 1.3229 - val_accuracy: 0.7781 - val_auc: 0.9771 - val_f1_score: 0.7558 - lr: 2.7091e-07
Epoch 38/40
583/583 [==============================] - ETA: 0s - loss: 1.7286 - accuracy: 0.7941 - auc: 0.8021 - f1_score: 0.6716
Epoch 38: val_loss did not improve from 1.32292
583/583 [==============================] - 87s 150ms/step - loss: 1.7286 - accuracy: 0.7941 - auc: 0.8021 - f1_score: 0.6716 - val_loss: 1.3245 - val_accuracy: 0.7791 - val_auc: 0.9772 - val_f1_score: 0.7569 - lr: 1.5300e-07
Epoch 39/40
583/583 [==============================] - ETA: 0s - loss: 1.7132 - accuracy: 0.8025 - auc: 0.8003 - f1_score: 0.6814
Epoch 39: val_loss did not improve from 1.32292
583/583 [==============================] - 88s 151ms/step - loss: 1.7132 - accuracy: 0.8025 - auc: 0.8003 - f1_score: 0.6814 - val_loss: 1.3243 - val_accuracy: 0.7781 - val_auc: 0.9773 - val_f1

In [12]:
phase2_eval_data = model.evaluate(
    test_ds,
    batch_size=BATCH_SIZE,
    return_dict=True,
    verbose=0
)
phase2_eval_data

{'loss': 1.3188482522964478,
 'accuracy': 0.7675568461418152,
 'auc': 0.9778870940208435,
 'f1_score': 0.7567267417907715}